# Limpieza del dataset AirPrice

Pipeline de limpieza del dataset de Inside Airbnb (Londres). Carga el dataset local desde `../data/listings.csv` y exporta los resultados a `../data/processed/`.

In [ ]:
import numpy as np
import pandas as pd


/kaggle/input/datasets/danpco/airprice-inside-airbnb-london/listings.csv/listings.csv


In [2]:
from pathlib import Path

ruta_local = Path("../data/listings.csv")
print("Dataset local existe:", ruta_local.exists())
print("Ruta:", ruta_local.resolve())

/kaggle/input/datasets/danpco/airprice-inside-airbnb-london/listings.csv/listings.csv


In [ ]:
import pandas as pd

ruta = "../data/listings.csv"

df = pd.read_csv(
    ruta,
    low_memory=False
)

print("Dataset cargado correctamente")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

✅ Dataset cargado correctamente
Filas: 92638
Columnas: 90


In [4]:
print("Cantidad de columnas:", len(df.columns))

for i, columna in enumerate(df.columns, start=1):
    print(i, columna)

Cantidad de columnas: 90
1 id
2 listing_url
3 scrape_id
4 last_scraped
5 source
6 name
7 description
8 neighborhood_overview
9 picture_url
10 host_id
11 host_url
12 host_profile_id
13 host_profile_url
14 host_name
15 host_since
16 hosts_time_as_user_years
17 hosts_time_as_user_months
18 hosts_time_as_host_years
19 hosts_time_as_host_months
20 host_location
21 host_about
22 host_response_time
23 host_response_rate
24 host_acceptance_rate
25 host_is_superhost
26 host_thumbnail_url
27 host_picture_url
28 host_neighbourhood
29 host_listings_count
30 host_total_listings_count
31 host_verifications
32 host_has_profile_pic
33 host_identity_verified
34 neighbourhood
35 neighbourhood_cleansed
36 neighbourhood_group_cleansed
37 latitude
38 longitude
39 property_type
40 room_type
41 accommodates
42 bathrooms
43 bathrooms_text
44 bedrooms
45 beds
46 amenities
47 price
48 price_quote_checkin_date
49 price_quote_checkout_date
50 price_quote_total_price
51 price_quote_price_per_night
52 price_quote_r

In [5]:
display(df.head())

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,11551,https://www.airbnb.com/rooms/11551,20260619061251,2026-06-25,city scrape,Artistic London Pied-à-Terre,Live like a local in this quintessential Londo...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,43039,...,4.84,4.54,4.49,NaN,NaN,1,1,0,0,0.99
1,13913,https://www.airbnb.com/rooms/13913,20260619061251,2026-06-25,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.88,4.79,4.79,NaN,NaN,2,1,1,0,0.30
2,15400,https://www.airbnb.com/rooms/15400,20260619061251,2026-06-26,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,NaN,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.94,4.74,NaN,NaN,1,1,0,0,0.49
3,17402,https://www.airbnb.com/rooms/17402,20260619061251,2026-06-29,city scrape,Spacious Fitzrovia 3bed/2bath By Soho & Oxford St,"Stay in this spacious 3-bedroom, 2-bath apartm...",NaN,https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.73,4.89,4.62,NaN,NaN,2,2,0,0,0.31
4,513198,https://www.airbnb.com/rooms/513198,20260619061251,2026-06-29,city scrape,Stunning views over Central London! (room),Wake up to the sound of Big Ben!<br /><br />Th...,NaN,https://a0.muscache.com/pictures/6442310/e4b3e...,2528030,...,4.74,4.72,4.42,NaN,NaN,2,1,1,0,0.25


## Inspección inicial

Verifica dimensiones, duplicados completos, IDs duplicados y la existencia de la variable `price`.

In [6]:
print("Filas totales:", len(df))
print("Filas duplicadas completas:", df.duplicated().sum())

Filas totales: 92638
Filas duplicadas completas: 0


In [7]:
print("IDs únicos:", df["id"].nunique())
print("IDs duplicados:", df["id"].duplicated().sum())

IDs únicos: 92638
IDs duplicados: 0


In [8]:
print("¿Existe price?:", "price" in df.columns)

display(df["price"].head(20))

¿Existe price?: True


0     $234.50
1     $127.00
2     $137.50
3     $606.67
4      $68.30
5      $49.06
6         NaN
7         NaN
8         NaN
9     $268.33
10     $63.44
11        NaN
12        NaN
13    $338.50
14     $37.40
15     $82.50
16        NaN
17        NaN
18     $45.00
19    $108.00
Name: price, dtype: object

## Exploración del precio

Analiza valores faltantes, el formato de `price` y su conversión a numérico (`price_clean`).

In [9]:
print(
    "Precios faltantes:",
    df["price"].isna().sum()
)

print(
    "Porcentaje de precios faltantes:",
    round(df["price"].isna().mean() * 100, 2),
    "%"
)

Precios faltantes: 30398
Porcentaje de precios faltantes: 32.81 %


In [10]:
faltantes = pd.DataFrame({
    "Columna": df.columns,
    "Faltantes": df.isnull().sum().values,
    "Porcentaje": (
        df.isnull().mean().values * 100
    )
})

faltantes = faltantes.sort_values(
    "Porcentaje",
    ascending=False
).reset_index(drop=True)

display(faltantes)

,Columna,Faltantes,Porcentaje
0,neighborhood_overview,92638,100.0
1,host_since,92638,100.0
2,host_response_time,92638,100.0
3,host_thumbnail_url,92638,100.0
4,host_acceptance_rate,92638,100.0
...,...,...,...
85,availability_30,0,0.0
86,calculated_host_listings_count,0,0.0
87,calculated_host_listings_count_entire_homes,0,0.0
88,calculated_host_listings_count_private_rooms,0,0.0


In [11]:
print("Tipo de dato de price:")
print(df["price"].dtype)

print("\nPrimeros 20 valores:")
display(df["price"].head(20))

Tipo de dato de price:
object

Primeros 20 valores:


0     $234.50
1     $127.00
2     $137.50
3     $606.67
4      $68.30
5      $49.06
6         NaN
7         NaN
8         NaN
9     $268.33
10     $63.44
11        NaN
12        NaN
13    $338.50
14     $37.40
15     $82.50
16        NaN
17        NaN
18     $45.00
19    $108.00
Name: price, dtype: object

In [12]:
df["price_clean"] = (
    df["price"]
    .astype(str)
    .str.replace("£", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price_clean"] = pd.to_numeric(
    df["price_clean"],
    errors="coerce"
)

In [13]:
display(
    df[
        ["price", "price_clean"]
    ].head(20)
)

,price,price_clean
0,$234.50,234.50
1,$127.00,127.00
2,$137.50,137.50
3,$606.67,606.67
4,$68.30,68.30
5,$49.06,49.06
6,NaN,NaN
7,NaN,NaN
8,NaN,NaN
9,$268.33,268.33


In [14]:
print(
    "Precios numéricos disponibles:",
    df["price_clean"].notna().sum()
)

print(
    "Precios faltantes después de conversión:",
    df["price_clean"].isna().sum()
)

Precios numéricos disponibles: 62240
Precios faltantes después de conversión: 30398


In [15]:
print(
    "Precios iguales a 0:",
    (df["price_clean"] == 0).sum()
)

print(
    "Precios menores que 0:",
    (df["price_clean"] < 0).sum()
)

Precios iguales a 0: 0
Precios menores que 0: 0


In [16]:
df["price_clean"].describe()

count     62240.000000
mean        271.469220
std        2190.599006
min           2.230000
25%         100.000000
50%         180.000000
75%         300.032500
max      527524.000000
Name: price_clean, dtype: float64

In [17]:
display(
    df[
        [
            "id",
            "price",
            "price_clean"
        ]
    ]
    .sort_values(
        "price_clean",
        ascending=False
    )
    .head(20)
)

,id,price,price_clean
67130,1389784758661918909,"$527,524.00",527524.0
8400,13254774,"$75,000.00",75000.0
61588,1289811437722717777,"$59,214.00",59214.0
57238,1203041287118705621,"$28,000.00",28000.0
56041,1183168933886992107,"$23,686.00",23686.0
57946,1207889725595849665,"$19,150.00",19150.0
64156,1342870516779702858,"$16,316.80",16316.8
56739,1196233560198207804,"$14,744.00",14744.0
66648,1384148619625788404,"$13,600.00",13600.0
34627,644043613421252064,"$13,209.00",13209.0


In [18]:
df["price_clean"].describe()

count     62240.000000
mean        271.469220
std        2190.599006
min           2.230000
25%         100.000000
50%         180.000000
75%         300.032500
max      527524.000000
Name: price_clean, dtype: float64

In [19]:
percentiles = df["price_clean"].quantile([
    0.50,
    0.75,
    0.90,
    0.95,
    0.97,
    0.98,
    0.99,
    0.995,
    0.999,
    1.00
])

print(percentiles)

0.500       180.00000
0.750       300.03250
0.900       488.02000
0.950       676.00000
0.970       854.92690
0.980      1011.53200
0.990      1373.45750
0.995      2008.18335
0.999      6234.06587
1.000    527524.00000
Name: price_clean, dtype: float64


In [20]:
columnas_revision = [
    "id",
    "name",
    "neighbourhood_cleansed",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "price",
    "price_clean"
]

columnas_revision = [
    col
    for col in columnas_revision
    if col in df.columns
]

top_precios = (
    df[columnas_revision]
    .sort_values(
        "price_clean",
        ascending=False
    )
    .head(30)
)

display(top_precios)

,id,name,neighbourhood_cleansed,room_type,accommodates,bedrooms,beds,price,price_clean
67130,1389784758661918909,Lux 2 Bed in Canary Wharf close to Excel & O2,Tower Hamlets,Entire home/apt,1,1.0,1.0,"$527,524.00",527524.00
8400,13254774,No Longer Available,Kensington and Chelsea,Private room,1,NaN,1.0,"$75,000.00",75000.00
61588,1289811437722717777,Central Room - Walk to Eye (KR),Lambeth,Private room,3,NaN,2.0,"$59,214.00",59214.00
57238,1203041287118705621,03. Short Walk to London Eye (TOK),Lambeth,Private room,3,NaN,1.0,"$28,000.00",28000.00
56041,1183168933886992107,Short Walk To London Eye (SUR),Lambeth,Private room,3,NaN,1.0,"$23,686.00",23686.00
57946,1207889725595849665,The Artists House,Westminster,Entire home/apt,16,22.0,28.0,"$19,150.00",19150.00
64156,1342870516779702858,Classic Double Room Near The Green Park LON,Westminster,Entire home/apt,2,1.0,1.0,"$16,316.80",16316.80
56739,1196233560198207804,London luxury.,Islington,Entire home/apt,16,6.0,7.0,"$14,744.00",14744.00
66648,1384148619625788404,Luxury One Bedroom Apartment Brompton Road LON,Westminster,Entire home/apt,2,1.0,2.0,"$13,600.00",13600.00
34627,644043613421252064,Grosvenor Penthouse,Westminster,Entire home/apt,10,5.0,5.0,"$13,209.00",13209.00


## Preparación del dataset

Crea `df_clean` como copia del original para aplicar la limpieza sin modificar el dataset fuente.

In [21]:
df_clean = df.copy()

print("Dataset original:", df.shape)
print("Copia para limpieza:", df_clean.shape)

Dataset original: (92638, 91)
Copia para limpieza: (92638, 91)


In [22]:
antes = len(df_clean)

df_clean = df_clean.dropna(
    subset=["price_clean"]
).copy()

despues = len(df_clean)

print("Registros antes:", antes)
print("Registros después:", despues)
print("Registros eliminados:", antes - despues)

Registros antes: 92638
Registros después: 62240
Registros eliminados: 30398


In [23]:
percentiles = df_clean["price_clean"].quantile([
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.97,
    0.98,
    0.99,
    0.995,
    0.999,
    1.00
])

display(percentiles)

0.250       100.00000
0.500       180.00000
0.750       300.03250
0.900       488.02000
0.950       676.00000
0.970       854.92690
0.980      1011.53200
0.990      1373.45750
0.995      2008.18335
0.999      6234.06587
1.000    527524.00000
Name: price_clean, dtype: float64

In [24]:
umbrales = [
    500,
    1000,
    2000,
    5000,
    10000,
    50000,
    100000
]

for umbral in umbrales:

    cantidad = (
        df_clean["price_clean"] > umbral
    ).sum()

    porcentaje = (
        cantidad / len(df_clean)
    ) * 100

    print(
        f"Precio > £{umbral:,}: "
        f"{cantidad} registros "
        f"({porcentaje:.3f}%)"
    )

Precio > £500: 5819 registros (9.349%)
Precio > £1,000: 1277 registros (2.052%)
Precio > £2,000: 317 registros (0.509%)
Precio > £5,000: 100 registros (0.161%)
Precio > £10,000: 21 registros (0.034%)
Precio > £50,000: 3 registros (0.005%)
Precio > £100,000: 1 registros (0.002%)


In [25]:
columnas_revision = [
    "id",
    "name",
    "neighbourhood_cleansed",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "price",
    "price_clean"
]

columnas_revision = [
    col
    for col in columnas_revision
    if col in df_clean.columns
]

display(
    df_clean[
        columnas_revision
    ]
    .sort_values(
        "price_clean",
        ascending=False
    )
    .head(30)
)

,id,name,neighbourhood_cleansed,room_type,accommodates,bedrooms,beds,price,price_clean
67130,1389784758661918909,Lux 2 Bed in Canary Wharf close to Excel & O2,Tower Hamlets,Entire home/apt,1,1.0,1.0,"$527,524.00",527524.00
8400,13254774,No Longer Available,Kensington and Chelsea,Private room,1,NaN,1.0,"$75,000.00",75000.00
61588,1289811437722717777,Central Room - Walk to Eye (KR),Lambeth,Private room,3,NaN,2.0,"$59,214.00",59214.00
57238,1203041287118705621,03. Short Walk to London Eye (TOK),Lambeth,Private room,3,NaN,1.0,"$28,000.00",28000.00
56041,1183168933886992107,Short Walk To London Eye (SUR),Lambeth,Private room,3,NaN,1.0,"$23,686.00",23686.00
57946,1207889725595849665,The Artists House,Westminster,Entire home/apt,16,22.0,28.0,"$19,150.00",19150.00
64156,1342870516779702858,Classic Double Room Near The Green Park LON,Westminster,Entire home/apt,2,1.0,1.0,"$16,316.80",16316.80
56739,1196233560198207804,London luxury.,Islington,Entire home/apt,16,6.0,7.0,"$14,744.00",14744.00
66648,1384148619625788404,Luxury One Bedroom Apartment Brompton Road LON,Westminster,Entire home/apt,2,1.0,2.0,"$13,600.00",13600.00
34627,644043613421252064,Grosvenor Penthouse,Westminster,Entire home/apt,10,5.0,5.0,"$13,209.00",13209.00


In [26]:
Q1 = df_clean["price_clean"].quantile(0.25)
Q3 = df_clean["price_clean"].quantile(0.75)

IQR = Q3 - Q1

limite_superior_iqr = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite superior IQR:", limite_superior_iqr)

print(
    "Registros por encima del límite:",
    (df_clean["price_clean"] > limite_superior_iqr).sum()
)

Q1: 100.0
Q3: 300.0325
IQR: 200.03250000000003
Límite superior IQR: 600.0812500000001
Registros por encima del límite: 3964


In [27]:
for i, columna in enumerate(df_clean.columns, start=1):
    print(f"{i}. {columna}")

1. id
2. listing_url
3. scrape_id
4. last_scraped
5. source
6. name
7. description
8. neighborhood_overview
9. picture_url
10. host_id
11. host_url
12. host_profile_id
13. host_profile_url
14. host_name
15. host_since
16. hosts_time_as_user_years
17. hosts_time_as_user_months
18. hosts_time_as_host_years
19. hosts_time_as_host_months
20. host_location
21. host_about
22. host_response_time
23. host_response_rate
24. host_acceptance_rate
25. host_is_superhost
26. host_thumbnail_url
27. host_picture_url
28. host_neighbourhood
29. host_listings_count
30. host_total_listings_count
31. host_verifications
32. host_has_profile_pic
33. host_identity_verified
34. neighbourhood
35. neighbourhood_cleansed
36. neighbourhood_group_cleansed
37. latitude
38. longitude
39. property_type
40. room_type
41. accommodates
42. bathrooms
43. bathrooms_text
44. bedrooms
45. beds
46. amenities
47. price
48. price_quote_checkin_date
49. price_quote_checkout_date
50. price_quote_total_price
51. price_quote_price_

In [28]:
variables_candidatas = [
    "id",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "review_scores_rating",
    "availability_365",
    "host_is_superhost",
    "instant_bookable",
    "amenities",
    "price_clean"
]

variables_existentes = [
    col
    for col in variables_candidatas
    if col in df_clean.columns
]

print("Variables candidatas existentes:")
print()

for variable in variables_existentes:
    print(variable)

Variables candidatas existentes:

id
neighbourhood_cleansed
latitude
longitude
property_type
room_type
accommodates
bathrooms
bedrooms
beds
minimum_nights
maximum_nights
number_of_reviews
reviews_per_month
review_scores_rating
availability_365
host_is_superhost
instant_bookable
amenities
price_clean


In [29]:
faltantes_modelo = pd.DataFrame({
    "Variable": variables_existentes,

    "Faltantes": [
        df_clean[col].isna().sum()
        for col in variables_existentes
    ],

    "Porcentaje_faltante": [
        df_clean[col].isna().mean() * 100
        for col in variables_existentes
    ]
})

faltantes_modelo = faltantes_modelo.sort_values(
    "Porcentaje_faltante",
    ascending=False
).reset_index(drop=True)

display(faltantes_modelo)

,Variable,Faltantes,Porcentaje_faltante
0,instant_bookable,62240,100.000000
1,bedrooms,13738,22.072622
2,review_scores_rating,13570,21.802699
3,reviews_per_month,13570,21.802699
4,bathrooms,6613,10.625000
5,beds,4282,6.879820
6,host_is_superhost,66,0.106041
7,minimum_nights,3,0.004820
8,maximum_nights,3,0.004820
9,latitude,0,0.000000


In [30]:
numericas_revision = [
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "review_scores_rating",
    "availability_365",
    "price_clean"
]

numericas_revision = [
    col
    for col in numericas_revision
    if col in df_clean.columns
]

display(
    df_clean[
        numericas_revision
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
accommodates,62240.0,3.548072,2.246596e+00,1.00,2.00,3.00,4.0000,1.600000e+01
bathrooms,55627.0,1.369793,7.369829e-01,0.50,1.00,1.00,1.5000,3.000000e+01
bedrooms,48502.0,1.762092,1.070337e+00,0.00,1.00,1.00,2.0000,3.000000e+01
beds,57958.0,2.022257,1.458038e+00,1.00,1.00,2.00,3.0000,5.000000e+01
minimum_nights,62237.0,3.611292,1.134052e+01,1.00,1.00,2.00,3.0000,3.650000e+02
maximum_nights,62237.0,319483.395761,2.590799e+07,1.00,365.00,365.00,1125.0000,2.147484e+09
number_of_reviews,62240.0,29.354579,6.242767e+01,0.00,1.00,7.00,30.0000,1.666000e+03
reviews_per_month,48670.0,1.257714,1.434635e+00,0.01,0.29,0.78,1.7000,2.717000e+01
review_scores_rating,48670.0,4.669419,5.111460e-01,1.00,4.57,4.82,5.0000,5.000000e+00
availability_365,62240.0,215.892947,1.195256e+02,0.00,99.00,241.00,330.0000,3.650000e+02


In [31]:
categoricas_revision = [
    "room_type",
    "property_type",
    "neighbourhood_cleansed",
    "host_is_superhost",
    "instant_bookable"
]

for columna in categoricas_revision:

    if columna in df_clean.columns:

        print("\n------------------------")
        print(columna)
        print("------------------------")

        print(
            df_clean[columna]
            .value_counts(
                dropna=False
            )
            .head(30)
        )


------------------------
room_type
------------------------
room_type
Entire home/apt    42123
Private room       19826
Shared room          183
Hotel room           108
Name: count, dtype: int64

------------------------
property_type
------------------------
property_type
Entire rental unit                    27154
Private room in home                   7461
Private room in rental unit            6975
Entire home                            6178
Entire condo                           5595
Private room in condo                  2224
Entire serviced apartment              1679
Room in hotel                          1060
Private room in townhouse               693
Entire townhouse                        584
Private room in bed and breakfast       435
Private room in guesthouse              306
Entire guesthouse                       207
Entire loft                             166
Private room in guest suite             154
Room in boutique hotel                  142
Entire guest suite  

In [32]:
print("=== DATASET FINAL ===")
print("Filas:", df_clean.shape[0])
print("Columnas:", df_clean.shape[1])

print("\n=== DUPLICADOS ===")
print("Duplicados:", df_clean.duplicated().sum())

print("\n=== PRECIO ===")
print("Precio mínimo:", df_clean["price_clean"].min())
print("Precio máximo:", df_clean["price_clean"].max())

print("\n=== RANGOS DE PRECIO ===")
print(df_clean["price_range"].value_counts())

print("\n=== PORCENTAJES ===")
print(
    df_clean["price_range"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

=== DATASET FINAL ===
Filas: 62240
Columnas: 91

=== DUPLICADOS ===
Duplicados: 0

=== PRECIO ===
Precio mínimo: 2.23
Precio máximo: 527524.0

=== RANGOS DE PRECIO ===


KeyError: 'price_range'

## Pipeline de limpieza paso a paso

Limpieza incremental sobre `df_final`: precio, duplicados, outliers, bathrooms, porcentajes del host, amenities y validaciones.

In [ ]:
import pandas as pd
import numpy as np
import ast
import json

# Partimos SIEMPRE del dataset original
df_final = df.copy()

print("Dataset original:")
print("Filas:", df_final.shape[0])
print("Columnas:", df_final.shape[1])

In [ ]:
auditoria = []

def registrar_paso(paso, antes, despues):
    auditoria.append({
        "Paso": paso,
        "Registros_antes": antes,
        "Registros_despues": despues,
        "Registros_eliminados": antes - despues
    })

In [ ]:
df_final["price_clean"] = (
    df_final["price"]
    .astype(str)
    .str.replace(r"[^\d.\-]", "", regex=True)
)

df_final["price_clean"] = pd.to_numeric(
    df_final["price_clean"],
    errors="coerce"
)

In [ ]:
print(df_final["price_clean"].describe())

In [ ]:
antes = len(df_final)

df_final = df_final.dropna(
    subset=["price_clean"]
).copy()

df_final = df_final[
    df_final["price_clean"] > 0
].copy()

registrar_paso(
    "Eliminar precio faltante o <= 0",
    antes,
    len(df_final)
)

print("Registros restantes:", len(df_final))

In [ ]:
antes = len(df_final)

df_final = df_final.drop_duplicates(
    subset=["id"],
    keep="first"
).copy()

registrar_paso(
    "Eliminar IDs duplicados",
    antes,
    len(df_final)
)

print("IDs duplicados:", df_final["id"].duplicated().sum())

In [ ]:
limite_inferior_precio = df_final["price_clean"].quantile(0.005)
limite_superior_precio = df_final["price_clean"].quantile(0.995)

print("Límite inferior:", limite_inferior_precio)
print("Límite superior:", limite_superior_precio)

In [ ]:
antes = len(df_final)

df_final = df_final[
    df_final["price_clean"].between(
        limite_inferior_precio,
        limite_superior_precio
    )
].copy()

registrar_paso(
    "Eliminar extremos de precio P0.5-P99.5",
    antes,
    len(df_final)
)

print("Registros restantes:", len(df_final))
print("Precio mínimo:", df_final["price_clean"].min())
print("Precio máximo:", df_final["price_clean"].max())

In [ ]:
if "bathrooms" in df_final.columns:

    df_final["bathrooms_num"] = pd.to_numeric(
        df_final["bathrooms"],
        errors="coerce"
    )

else:

    df_final["bathrooms_num"] = np.nan

In [ ]:
if "bathrooms_text" in df_final.columns:

    bathrooms_text_num = pd.to_numeric(
        df_final["bathrooms_text"]
        .astype(str)
        .str.extract(r"(\d+(?:\.\d+)?)")[0],
        errors="coerce"
    )

    df_final["bathrooms_num"] = (
        df_final["bathrooms_num"]
        .fillna(bathrooms_text_num)
    )

In [ ]:
def convertir_porcentaje(serie):

    return pd.to_numeric(
        serie
        .astype(str)
        .str.replace("%", "", regex=False),
        errors="coerce"
    )

In [ ]:
if "host_response_rate" in df_final.columns:
    df_final["host_response_rate_num"] = convertir_porcentaje(
        df_final["host_response_rate"]
    )

if "host_acceptance_rate" in df_final.columns:
    df_final["host_acceptance_rate_num"] = convertir_porcentaje(
        df_final["host_acceptance_rate"]
    )

In [ ]:
def contar_amenities(valor):

    if pd.isna(valor):
        return 0

    try:

        lista = ast.literal_eval(valor)

        if isinstance(lista, list):
            return len(lista)

    except:
        pass

    texto = str(valor).strip()

    if texto in ["", "[]"]:
        return 0

    return len([
        elemento
        for elemento in texto.strip("[]").split(",")
        if elemento.strip()
    ])

In [ ]:
if "amenities" in df_final.columns:

    df_final["amenities_count"] = (
        df_final["amenities"]
        .apply(contar_amenities)
    )

In [ ]:
if "number_of_reviews" in df_final.columns:

    df_final["has_reviews"] = (
        df_final["number_of_reviews"]
        .fillna(0)
        .gt(0)
        .astype(int)
    )

In [ ]:
antes = len(df_final)

if "accommodates" in df_final.columns:

    df_final = df_final[
        df_final["accommodates"].notna()
        &
        (df_final["accommodates"] > 0)
    ].copy()

registrar_paso(
    "Eliminar capacidad inválida",
    antes,
    len(df_final)
)

In [ ]:
antes = len(df_final)

df_final = df_final[
    df_final["latitude"].between(-90, 90)
    &
    df_final["longitude"].between(-180, 180)
].copy()

registrar_paso(
    "Eliminar coordenadas inválidas",
    antes,
    len(df_final)
)

In [ ]:
columnas_no_negativas = [
    "bathrooms_num",
    "bedrooms",
    "beds",
    "number_of_reviews",
    "reviews_per_month"
]

for columna in columnas_no_negativas:

    if columna in df_final.columns:

        df_final.loc[
            df_final[columna] < 0,
            columna
        ] = np.nan

In [ ]:
if "minimum_nights" in df_final.columns:

    df_final.loc[
        df_final["minimum_nights"] < 1,
        "minimum_nights"
    ] = np.nan

In [ ]:
if "availability_365" in df_final.columns:

    df_final.loc[
        ~df_final["availability_365"].between(0, 365),
        "availability_365"
    ] = np.nan

In [ ]:
if "review_scores_rating" in df_final.columns:

    df_final.loc[
        ~df_final["review_scores_rating"].between(0, 5),
        "review_scores_rating"
    ] = np.nan

In [ ]:
for columna in [
    "host_response_rate_num",
    "host_acceptance_rate_num"
]:

    if columna in df_final.columns:

        df_final.loc[
            ~df_final[columna].between(0, 100),
            columna
        ] = np.nan

In [ ]:
categoricas = [
    "neighbourhood_cleansed",
    "property_type",
    "room_type",
    "host_is_superhost",
    "host_identity_verified",
    "instant_bookable"
]

In [ ]:
categoricas = [
    c
    for c in categoricas
    if c in df_final.columns
]

In [ ]:
for columna in categoricas:

    df_final[columna] = (
        df_final[columna]
        .astype("string")
        .str.strip()
        .fillna("UNKNOWN")
    )

In [ ]:
if "property_type" in df_final.columns:

    frecuencia = (
        df_final["property_type"]
        .value_counts(normalize=True)
    )

    categorias_raras = frecuencia[
        frecuencia < 0.0025
    ].index

    df_final["property_type"] = (
        df_final["property_type"]
        .replace(
            categorias_raras,
            "OTHER"
        )
    )

In [ ]:
variables_finales = [
    "id",

    # ubicación
    "neighbourhood_cleansed",
    "latitude",
    "longitude",

    # alojamiento
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms_num",
    "bedrooms",
    "beds",

    # estancia
    "minimum_nights",
    "availability_365",

    # reseñas
    "number_of_reviews",
    "reviews_per_month",
    "review_scores_rating",
    "has_reviews",

    # host
    "host_is_superhost",
    "host_identity_verified",
    "instant_bookable",
    "host_response_rate_num",
    "host_acceptance_rate_num",

    # características
    "amenities_count",

    # precio
    "price_clean"
]

In [ ]:
variables_finales = [
    columna
    for columna in variables_finales
    if columna in df_final.columns
]

df_final = df_final[
    variables_finales
].copy()

print("Variables finales:", len(df_final.columns))
print(df_final.columns.tolist())

In [ ]:
numericas_con_faltantes = [
    "bathrooms_num",
    "bedrooms",
    "beds",
    "minimum_nights",
    "availability_365",
    "reviews_per_month",
    "review_scores_rating",
    "host_response_rate_num",
    "host_acceptance_rate_num"
]

numericas_con_faltantes = [
    c
    for c in numericas_con_faltantes
    if c in df_final.columns
]

In [ ]:
for columna in numericas_con_faltantes:

    df_final[
        f"{columna}_missing"
    ] = (
        df_final[columna]
        .isna()
        .astype(int)
    )

In [ ]:
for columna in numericas_con_faltantes:

    mediana = df_final[
        columna
    ].median()

    df_final[
        columna
    ] = (
        df_final[columna]
        .fillna(mediana)
    )

In [ ]:
for columna in categoricas:

    if columna in df_final.columns:

        df_final[columna] = (
            df_final[columna]
            .fillna("UNKNOWN")
        )

In [ ]:
faltantes_finales = (
    df_final
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(faltantes_finales)

In [ ]:
print(
    "TOTAL DE VALORES FALTANTES:",
    df_final.isna().sum().sum()
)

## Pipeline completo consolidado

Versión consolidada de todo el proceso: limpia, imputa faltantes, crea `price_range` y exporta `airprice_clean_local.csv`, la auditoría y la metadata.

In [ ]:


import pandas as pd
import numpy as np
import ast
import json
from pathlib import Path



pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)


df_final = df.copy()

print("=" * 70)
print("AIRPRICE - PROCESO COMPLETO DE LIMPIEZA")
print("=" * 70)

print("\nDataset original:")
print("Filas:", df_final.shape[0])
print("Columnas:", df_final.shape[1])


# ------------------------------------------------------------
# 1. AUDITORÍA
# ------------------------------------------------------------

auditoria = []

def registrar_paso(paso, antes, despues):
    auditoria.append({
        "Paso": paso,
        "Registros_antes": int(antes),
        "Registros_despues": int(despues),
        "Registros_eliminados": int(antes - despues)
    })


# ------------------------------------------------------------
# 2. LIMPIAR VARIABLE PRICE
# ------------------------------------------------------------

df_final["price_clean"] = (
    df_final["price"]
    .astype("string")
    .str.replace(r"[^\d.\-]", "", regex=True)
)

df_final["price_clean"] = pd.to_numeric(
    df_final["price_clean"],
    errors="coerce"
)

antes = len(df_final)

df_final = df_final[
    df_final["price_clean"].notna()
    &
    (df_final["price_clean"] > 0)
].copy()

registrar_paso(
    "Eliminar registros sin precio válido",
    antes,
    len(df_final)
)

print("\n1. Precio limpio")
print("Registros restantes:", len(df_final))


# ------------------------------------------------------------
# 3. ELIMINAR DUPLICADOS POR ID
# ------------------------------------------------------------

antes = len(df_final)

df_final = df_final.drop_duplicates(
    subset=["id"],
    keep="first"
).copy()

registrar_paso(
    "Eliminar IDs duplicados",
    antes,
    len(df_final)
)

print("\n2. Duplicados")
print("IDs duplicados:", df_final["id"].duplicated().sum())


# ------------------------------------------------------------
# 4. TRATAMIENTO DE PRECIOS EXTREMOS
# ------------------------------------------------------------
# Conservamos el 99% central del dataset.
# Eliminamos únicamente 0.5% inferior y 0.5% superior.

p005 = df_final["price_clean"].quantile(0.005)
p995 = df_final["price_clean"].quantile(0.995)

print("\n3. Límites de precio utilizados")
print("Percentil 0.5%:", round(p005, 2))
print("Percentil 99.5%:", round(p995, 2))

antes = len(df_final)

df_final = df_final[
    df_final["price_clean"].between(
        p005,
        p995,
        inclusive="both"
    )
].copy()

registrar_paso(
    "Eliminar extremos de precio P0.5-P99.5",
    antes,
    len(df_final)
)

print("Registros restantes:", len(df_final))
print("Precio mínimo:", df_final["price_clean"].min())
print("Precio máximo:", df_final["price_clean"].max())


# ------------------------------------------------------------
# 5. CREAR BATHROOMS_NUM
# ------------------------------------------------------------

df_final["bathrooms_num"] = np.nan

if "bathrooms" in df_final.columns:
    df_final["bathrooms_num"] = pd.to_numeric(
        df_final["bathrooms"],
        errors="coerce"
    )

if "bathrooms_text" in df_final.columns:

    bathrooms_desde_texto = pd.to_numeric(
        df_final["bathrooms_text"]
        .astype("string")
        .str.extract(r"(\d+(?:\.\d+)?)")[0],
        errors="coerce"
    )

    df_final["bathrooms_num"] = (
        df_final["bathrooms_num"]
        .fillna(bathrooms_desde_texto)
    )


# ------------------------------------------------------------
# 6. CONVERTIR PORCENTAJES DEL HOST
# ------------------------------------------------------------

def convertir_porcentaje(serie):
    return pd.to_numeric(
        serie
        .astype("string")
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

if "host_response_rate" in df_final.columns:
    df_final["host_response_rate_num"] = convertir_porcentaje(
        df_final["host_response_rate"]
    )

if "host_acceptance_rate" in df_final.columns:
    df_final["host_acceptance_rate_num"] = convertir_porcentaje(
        df_final["host_acceptance_rate"]
    )


# ------------------------------------------------------------
# 7. PROCESAR AMENITIES
# ------------------------------------------------------------

def contar_amenities(valor):

    if pd.isna(valor):
        return 0

    try:
        lista = ast.literal_eval(str(valor))

        if isinstance(lista, list):
            return len(lista)

    except (ValueError, SyntaxError):
        pass

    texto = str(valor).strip()

    if texto in ["", "[]", "nan", "<NA>"]:
        return 0

    elementos = [
        elemento
        for elemento in texto.strip("[]").split(",")
        if elemento.strip()
    ]

    return len(elementos)


if "amenities" in df_final.columns:
    df_final["amenities_count"] = (
        df_final["amenities"]
        .apply(contar_amenities)
    )
else:
    df_final["amenities_count"] = 0


# ------------------------------------------------------------
# 8. CREAR HAS_REVIEWS
# ------------------------------------------------------------

if "number_of_reviews" in df_final.columns:

    df_final["number_of_reviews"] = pd.to_numeric(
        df_final["number_of_reviews"],
        errors="coerce"
    )

    df_final["has_reviews"] = (
        df_final["number_of_reviews"]
        .fillna(0)
        .gt(0)
        .astype(int)
    )

else:
    df_final["has_reviews"] = 0


# ------------------------------------------------------------
# 9. CONVERTIR VARIABLES NUMÉRICAS
# ------------------------------------------------------------

variables_numericas_base = [
    "latitude",
    "longitude",
    "accommodates",
    "bedrooms",
    "beds",
    "minimum_nights",
    "availability_365",
    "number_of_reviews",
    "reviews_per_month",
    "review_scores_rating",
    "bathrooms_num",
    "host_response_rate_num",
    "host_acceptance_rate_num",
    "amenities_count"
]

for columna in variables_numericas_base:

    if columna in df_final.columns:
        df_final[columna] = pd.to_numeric(
            df_final[columna],
            errors="coerce"
        )


# ------------------------------------------------------------
# 10. ELIMINAR CAPACIDAD INVÁLIDA
# ------------------------------------------------------------

if "accommodates" in df_final.columns:

    antes = len(df_final)

    df_final = df_final[
        df_final["accommodates"].notna()
        &
        (df_final["accommodates"] > 0)
    ].copy()

    registrar_paso(
        "Eliminar capacidad inválida",
        antes,
        len(df_final)
    )


# ------------------------------------------------------------
# 11. VALIDAR COORDENADAS
# ------------------------------------------------------------

if (
    "latitude" in df_final.columns
    and "longitude" in df_final.columns
):

    antes = len(df_final)

    condicion_coordenadas = (
        df_final["latitude"].between(-90, 90)
        &
        df_final["longitude"].between(-180, 180)
    )

    df_final = df_final[
        condicion_coordenadas
    ].copy()

    registrar_paso(
        "Eliminar coordenadas inválidas",
        antes,
        len(df_final)
    )


# ------------------------------------------------------------
# 12. CORREGIR VALORES NUMÉRICOS IMPOSIBLES
# ------------------------------------------------------------

no_negativas = [
    "bathrooms_num",
    "bedrooms",
    "beds",
    "number_of_reviews",
    "reviews_per_month",
    "amenities_count"
]

for columna in no_negativas:

    if columna in df_final.columns:

        df_final.loc[
            df_final[columna] < 0,
            columna
        ] = np.nan


# Minimum nights debe ser >= 1

if "minimum_nights" in df_final.columns:

    df_final.loc[
        df_final["minimum_nights"] < 1,
        "minimum_nights"
    ] = np.nan


# Availability debe estar entre 0 y 365

if "availability_365" in df_final.columns:

    df_final.loc[
        ~df_final["availability_365"].between(0, 365),
        "availability_365"
    ] = np.nan


# Rating de Airbnb normalmente está entre 0 y 5

if "review_scores_rating" in df_final.columns:

    df_final.loc[
        ~df_final["review_scores_rating"].between(0, 5),
        "review_scores_rating"
    ] = np.nan


# Tasas entre 0 y 100

for columna in [
    "host_response_rate_num",
    "host_acceptance_rate_num"
]:

    if columna in df_final.columns:

        df_final.loc[
            ~df_final[columna].between(0, 100),
            columna
        ] = np.nan


# ------------------------------------------------------------
# 13. LIMPIEZA DE VARIABLES CATEGÓRICAS
# ------------------------------------------------------------

categoricas_base = [
    "neighbourhood_cleansed",
    "property_type",
    "room_type",
    "host_is_superhost",
    "host_identity_verified",
    "instant_bookable"
]

categoricas_existentes = [
    columna
    for columna in categoricas_base
    if columna in df_final.columns
]

for columna in categoricas_existentes:

    # Convertir a string
    df_final[columna] = (
        df_final[columna]
        .astype("string")
        .str.strip()
    )

    # Cadenas vacías pasan a NA
    df_final[columna] = (
        df_final[columna]
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA
            }
        )
    )

    # Categoría explícita para faltantes
    df_final[columna] = (
        df_final[columna]
        .fillna("UNKNOWN")
    )


# ------------------------------------------------------------
# 14. AGRUPAR PROPERTY_TYPE RAROS
# ------------------------------------------------------------

if "property_type" in df_final.columns:

    frecuencia_property = (
        df_final["property_type"]
        .value_counts(normalize=True)
    )

    categorias_raras = frecuencia_property[
        frecuencia_property < 0.0025
    ].index

    df_final["property_type"] = (
        df_final["property_type"]
        .replace(
            categorias_raras,
            "OTHER"
        )
    )


# ------------------------------------------------------------
# 15. SELECCIONAR VARIABLES DEFINITIVAS
# ------------------------------------------------------------

variables_finales = [
    "id",

    # Ubicación
    "neighbourhood_cleansed",
    "latitude",
    "longitude",

    # Tipo y capacidad
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms_num",
    "bedrooms",
    "beds",

    # Estancia / disponibilidad
    "minimum_nights",
    "availability_365",

    # Reseñas
    "number_of_reviews",
    "reviews_per_month",
    "review_scores_rating",
    "has_reviews",

    # Características del host
    "host_is_superhost",
    "host_identity_verified",
    "instant_bookable",
    "host_response_rate_num",
    "host_acceptance_rate_num",

    # Servicios
    "amenities_count",

    # Precio
    "price_clean"
]

variables_finales = [
    columna
    for columna in variables_finales
    if columna in df_final.columns
]

df_final = df_final[
    variables_finales
].copy()

print("\n4. Variables seleccionadas")
print("Cantidad:", len(df_final.columns))
print(df_final.columns.tolist())


# ------------------------------------------------------------
# 16. IDENTIFICAR AUTOMÁTICAMENTE NUMÉRICAS Y CATEGÓRICAS
# ------------------------------------------------------------

columnas_numericas = (
    df_final
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

columnas_categoricas = [
    columna
    for columna in df_final.columns
    if columna not in columnas_numericas
]


# NO imputamos id ni price_clean:
columnas_numericas_imputables = [
    columna
    for columna in columnas_numericas
    if columna not in [
        "id",
        "price_clean"
    ]
]


# ------------------------------------------------------------
# 17. CREAR BANDERAS DE FALTANTES
# ------------------------------------------------------------

columnas_con_nan_original = [
    columna
    for columna in columnas_numericas_imputables
    if df_final[columna].isna().any()
]

for columna in columnas_con_nan_original:

    df_final[
        f"{columna}_missing"
    ] = (
        df_final[columna]
        .isna()
        .astype("int8")
    )

print("\nVariables numéricas con faltantes encontradas:")
print(columnas_con_nan_original)


# ------------------------------------------------------------
# 18. IMPUTAR TODAS LAS VARIABLES NUMÉRICAS
# ------------------------------------------------------------

for columna in columnas_numericas_imputables:

    if df_final[columna].isna().any():

        mediana = df_final[columna].median()

        # Por seguridad, si toda la columna fuera NaN
        if pd.isna(mediana):
            mediana = 0

        df_final[columna] = (
            df_final[columna]
            .fillna(mediana)
        )


# ------------------------------------------------------------
# 19. IMPUTAR TODAS LAS VARIABLES CATEGÓRICAS
# ------------------------------------------------------------

for columna in columnas_categoricas:

    df_final[columna] = (
        df_final[columna]
        .astype("string")
        .fillna("UNKNOWN")
    )

    df_final[columna] = (
        df_final[columna]
        .replace(
            {
                "": "UNKNOWN",
                "nan": "UNKNOWN",
                "None": "UNKNOWN",
                "<NA>": "UNKNOWN"
            }
        )
    )


# ------------------------------------------------------------
# 20. COMPROBACIÓN DE FALTANTES ANTES DEL TARGET
# ------------------------------------------------------------

total_nan = int(
    df_final
    .isna()
    .sum()
    .sum()
)

print("\n5. Valores faltantes después de imputación:")
print(total_nan)

if total_nan > 0:

    print("\nColumnas que todavía tienen faltantes:")

    print(
        df_final
        .isna()
        .sum()
        .loc[
            lambda x: x > 0
        ]
        .sort_values(ascending=False)
    )

    # Segundo respaldo automático.
    # Cualquier numérica restante -> mediana o 0
    for columna in df_final.columns:

        if df_final[columna].isna().any():

            if pd.api.types.is_numeric_dtype(
                df_final[columna]
            ):

                mediana = df_final[
                    columna
                ].median()

                if pd.isna(mediana):
                    mediana = 0

                df_final[columna] = (
                    df_final[columna]
                    .fillna(mediana)
                )

            else:

                df_final[columna] = (
                    df_final[columna]
                    .astype("string")
                    .fillna("UNKNOWN")
                )


# ------------------------------------------------------------
# 21. CREAR RANGO DE PRECIOS DEFINITIVO
# ------------------------------------------------------------

df_final = df_final.drop(
    columns=["price_range"],
    errors="ignore"
)

etiquetas = [
    "ECONOMICO",
    "MEDIO",
    "ALTO",
    "PREMIUM"
]

df_final["price_range"], limites_precio = pd.qcut(
    df_final["price_clean"],
    q=4,
    labels=etiquetas,
    retbins=True
)


# ------------------------------------------------------------
# 22. MOSTRAR LOS RANGOS OBTENIDOS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RANGOS DE PRECIO")
print("=" * 70)

for i, etiqueta in enumerate(etiquetas):

    print(
        f"{etiqueta}: "
        f"£{limites_precio[i]:.2f} "
        f"→ "
        f"£{limites_precio[i + 1]:.2f}"
    )


# ------------------------------------------------------------
# 23. BALANCE DE CLASES
# ------------------------------------------------------------

balance = pd.DataFrame({
    "Cantidad":
        df_final["price_range"]
        .value_counts(),

    "Porcentaje":
        df_final["price_range"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
})

print("\n" + "=" * 70)
print("BALANCE DEL DATASET")
print("=" * 70)

display(balance)


# ------------------------------------------------------------
# 24. AUDITORÍA FINAL DE VALORES FALTANTES
# ------------------------------------------------------------

faltantes_finales = (
    df_final
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n" + "=" * 70)
print("VALORES FALTANTES POR COLUMNA")
print("=" * 70)

print(
    faltantes_finales[
        faltantes_finales > 0
    ]
)

total_faltantes_final = int(
    df_final
    .isna()
    .sum()
    .sum()
)

print(
    "\nTOTAL DE VALORES FALTANTES:",
    total_faltantes_final
)


# ------------------------------------------------------------
# 25. VALIDACIÓN FINAL DEL DATASET
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDACIÓN FINAL AIRPRICE")
print("=" * 70)

print("Filas finales:", len(df_final))
print("Columnas finales:", len(df_final.columns))

print(
    "IDs únicos:",
    df_final["id"].nunique()
)

print(
    "IDs duplicados:",
    df_final["id"]
    .duplicated()
    .sum()
)

print(
    "Valores faltantes:",
    total_faltantes_final
)

print(
    "Precio mínimo:",
    df_final["price_clean"].min()
)

print(
    "Precio máximo:",
    df_final["price_clean"].max()
)

print(
    "Cantidad de clases:",
    df_final["price_range"].nunique()
)

print("\nDistribución de clases:")

print(
    df_final["price_range"]
    .value_counts()
)

print("\nPorcentajes:")

print(
    df_final["price_range"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ------------------------------------------------------------
# 26. VALIDACIONES AUTOMÁTICAS
# ------------------------------------------------------------

assert len(df_final) > 35000, (
    "ERROR: El dataset final tiene menos "
    "de 35.000 registros."
)

assert df_final["id"].duplicated().sum() == 0, (
    "ERROR: Existen IDs duplicados."
)

assert df_final.isna().sum().sum() == 0, (
    "ERROR: Todavía existen valores faltantes."
)

assert df_final["price_range"].nunique() == 4, (
    "ERROR: No existen las cuatro clases esperadas."
)

assert (df_final["price_clean"] > 0).all(), (
    "ERROR: Existen precios iguales o menores que cero."
)

assert df_final[
    "latitude"
].between(-90, 90).all(), (
    "ERROR: Existen latitudes inválidas."
)

assert df_final[
    "longitude"
].between(-180, 180).all(), (
    "ERROR: Existen longitudes inválidas."
)

print("\n" + "=" * 70)
print("✅ TODAS LAS VALIDACIONES SUPERADAS")
print("=" * 70)


# ------------------------------------------------------------
# 27. CREAR AUDITORÍA DE LIMPIEZA
# ------------------------------------------------------------

auditoria_df = pd.DataFrame(
    auditoria
)

print("\nAUDITORÍA DEL PROCESO:")
display(auditoria_df)


# ------------------------------------------------------------
# 28. CREAR METADATA
# ------------------------------------------------------------

metadata = {
    "proyecto": "AirPrice",

    "fuente": "Inside Airbnb - London",

    "registros_originales": int(
        len(df)
    ),

    "registros_finales": int(
        len(df_final)
    ),

    "columnas_finales": int(
        len(df_final.columns)
    ),

    "precio_minimo": float(
        df_final["price_clean"].min()
    ),

    "precio_maximo": float(
        df_final["price_clean"].max()
    ),

    "tratamiento_outliers": {
        "metodo": "Recorte por percentiles",
        "percentil_inferior": 0.5,
        "percentil_superior": 99.5,
        "limite_inferior": float(p005),
        "limite_superior": float(p995)
    },

    "limites_clases": {
        etiquetas[i]: {
            "min": float(
                limites_precio[i]
            ),
            "max": float(
                limites_precio[i + 1]
            )
        }
        for i in range(4)
    },

    "clases": etiquetas,

    "valores_faltantes_finales": int(
        df_final
        .isna()
        .sum()
        .sum()
    )
}


# ------------------------------------------------------------
# 29. EXPORTAR DATASET FINAL
# ------------------------------------------------------------

ruta_dataset = (
    "../data/processed/"
    "airprice_clean_local.csv"
)

df_final.to_csv(
    ruta_dataset,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 30. EXPORTAR AUDITORÍA
# ------------------------------------------------------------

ruta_auditoria = (
    "../data/processed/"
    "airprice_cleaning_audit_local.csv"
)

auditoria_df.to_csv(
    ruta_auditoria,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 31. EXPORTAR METADATA
# ------------------------------------------------------------

ruta_metadata = (
    "../data/processed/"
    "airprice_metadata_local.json"
)

with open(
    ruta_metadata,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        metadata,
        archivo,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 32. COMPROBAR QUE LOS ARCHIVOS EXISTEN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)

for ruta in [
    ruta_dataset,
    ruta_auditoria,
    ruta_metadata
]:
    print(
        "" if Path(ruta).exists() else "",
        ruta
    )


# ------------------------------------------------------------
# 33. RESUMEN FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESUMEN FINAL")
print("=" * 70)

print(
    f"""
Dataset original:
{len(df):,} registros

Dataset final:
{len(df_final):,} registros

Registros eliminados:
{len(df) - len(df_final):,}

Variables finales:
{len(df_final.columns)}

Duplicados:
{df_final["id"].duplicated().sum()}

Valores faltantes:
{df_final.isna().sum().sum()}

Precio mínimo:
£{df_final["price_clean"].min():,.2f}

Precio máximo:
£{df_final["price_clean"].max():,.2f}

Clases:
{df_final["price_range"].nunique()}

Archivo final:
airprice_clean_local.csv
"""
)

print(
    "LIMPIEZA DEL DATASET "
    "COMPLETADA CORRECTAMENTE"
)

## Verificación de resultados

CSV exportado, valida filas, valores faltantes y balance de clases.

In [ ]:
df_verificacion = pd.read_csv(
    "../data/processed/airprice_clean_local.csv",
    low_memory=False
)

print("Filas guardadas:", df_verificacion.shape[0])
print("Columnas guardadas:", df_verificacion.shape[1])
print("Faltantes:", df_verificacion.isna().sum().sum())
print("Duplicados ID:", df_verificacion["id"].duplicated().sum())
print("\nClases:")
print(df_verificacion["price_range"].value_counts())